In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("cleaned_orders.xlsx")


# Numeric conversions

numeric_cols = [
    "quantity",
    "unit_price",
    "discount",
    "sales",
    "cost",
    "profit"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# cleaned_discount

df["cleaned_discount"] = df["discount"].fillna(0)


# calculated_sales

df["calculated_sales"] = (
    df["quantity"]
    * df["unit_price"]
    * (1 - df["cleaned_discount"])
)


# calculated_profit

df["calculated_profit"] = (
    df["calculated_sales"]
    - df["cost"]
)


# profit_margin

df["profit_margin"] = np.where(
    df["calculated_sales"] != 0,
    df["calculated_profit"] / df["calculated_sales"],
    np.nan
)


# Date columns

df["order_date"] = pd.to_datetime(df["order_date"])

# order_month
df["order_month"] = df["order_date"].dt.month_name()

# order_year
df["order_year"] = df["order_date"].dt.year


# Data quality flag

df["data_quality_flag"] = "Clean"

# Warning
df.loc[
    (
        df["cancelled_order_flag"]
        | df["failed_payment_flag"]
        | df["refunded_order_flag"]
    ),
    "data_quality_flag"
] = "Warning"

# Invalid
df.loc[
    (
        df["invalid_discount_flag"]
        | df["invalid_shipping_flag"]
    ),
    "data_quality_flag"
] = "Invalid"


# Sales mismatch flag

df["sales_difference"] = (
    df["sales"]
    - df["calculated_sales"]
).round(2)

df["profit_difference"] = (
    df["profit"]
    - df["calculated_profit"]
).round(2)


# Save

df.to_excel(
    "cleaned_orders.xlsx",
    index=False
)

print("="*50)
print("Clean records :",
      (df["data_quality_flag"] == "Clean").sum())

print("Warning records :",
      (df["data_quality_flag"] == "Warning").sum())

print("Invalid records :",
      (df["data_quality_flag"] == "Invalid").sum())

print("="*50)
print("Task 6 Completed")

Clean records : 612
Warning records : 187
Invalid records : 113
Task 6 Completed
